# Lab 7: Deploy RAG as Azure Function Endpoint

**Duration:** 35 minutes  
**Environment:** Azure ML Studio Notebook  

---

## Learning Objectives

- Package the RAG pipeline as an Azure Function
- Deploy a serverless REST API for the banking chatbot
- Test the endpoint with requests
- Configure app settings securely

## Prerequisites

Labs 1-3 completed

---

© 2026 Great Learning. All rights reserved.

## Step 1: Load Configuration

In [ ]:
import os, json, urllib.request, ssl, subprocess

RESOURCE_GROUP = 'rg-promptflow-rag-lab'

# Auto-detect OpenAI resource that has BOTH gpt-4o and text-embedding-3-small deployments
# This handles the case where Lab 1 was run multiple times creating multiple OpenAI resources
print('Detecting Azure resources...')
r = subprocess.run(['az', 'cognitiveservices', 'account', 'list', '-g', RESOURCE_GROUP,
    '--query', "[?kind=='OpenAI'].name", '-o', 'tsv'], capture_output=True, text=True)
openai_candidates = [n.strip() for n in r.stdout.strip().split('\n') if n.strip()]

OPENAI_NAME = ''
for candidate in openai_candidates:
    # Check if this resource has both required deployments
    r = subprocess.run(['az', 'cognitiveservices', 'account', 'deployment', 'list',
        '-n', candidate, '-g', RESOURCE_GROUP, '--query', '[].name', '-o', 'tsv'],
        capture_output=True, text=True)
    deployments = r.stdout.strip().split('\n')
    if 'gpt-4o' in deployments and 'text-embedding-3-small' in deployments:
        OPENAI_NAME = candidate
        print(f'  Found OpenAI with deployments: {candidate}')
        break

if not OPENAI_NAME:
    print('ERROR: No OpenAI resource found with both gpt-4o and text-embedding-3-small deployments.')
    print(f'  Candidates checked: {openai_candidates}')
    print('  Fix: Go back to Lab 1 and run Steps 4 + 5 to create deployments.')
    raise SystemExit(1)

# Auto-detect Search service
r = subprocess.run(['az', 'search', 'service', 'list', '-g', RESOURCE_GROUP,
    '--query', '[].name', '-o', 'tsv'], capture_output=True, text=True)
search_candidates = [n.strip() for n in r.stdout.strip().split('\n') if n.strip()]
SEARCH_NAME = search_candidates[0] if search_candidates else ''

# Auto-detect Storage account
r = subprocess.run(['az', 'storage', 'account', 'list', '-g', RESOURCE_GROUP,
    '--query', '[0].name', '-o', 'tsv'], capture_output=True, text=True)
STORAGE_NAME = r.stdout.strip()

if not SEARCH_NAME:
    print('ERROR: No Search service found. Run Lab 1 first.')
    raise SystemExit(1)

# Get endpoints and keys
r = subprocess.run(['az', 'cognitiveservices', 'account', 'show', '-n', OPENAI_NAME, '-g', RESOURCE_GROUP,
    '--query', 'properties.endpoint', '-o', 'tsv'], capture_output=True, text=True)
OPENAI_ENDPOINT = r.stdout.strip()
if not OPENAI_ENDPOINT.endswith('/'): OPENAI_ENDPOINT += '/'

r = subprocess.run(['az', 'cognitiveservices', 'account', 'keys', 'list', '-n', OPENAI_NAME, '-g', RESOURCE_GROUP,
    '--query', 'key1', '-o', 'tsv'], capture_output=True, text=True)
OPENAI_KEY = r.stdout.strip()

SEARCH_ENDPOINT = f'https://{SEARCH_NAME}.search.windows.net'
r = subprocess.run(['az', 'search', 'admin-key', 'show', '--service-name', SEARCH_NAME, '-g', RESOURCE_GROUP,
    '--query', 'primaryKey', '-o', 'tsv'], capture_output=True, text=True)
SEARCH_KEY = r.stdout.strip()

ctx = ssl.create_default_context()

print(f'\n✅ Resources detected:')
print(f'  OpenAI:  {OPENAI_NAME}')
print(f'  Search:  {SEARCH_NAME}')
print(f'  Storage: {STORAGE_NAME}')
print(f'  Endpoint: {OPENAI_ENDPOINT}')

## Azure CLI Authentication

Azure ML compute instances have a **managed identity** — this logs in instantly with no browser needed.

> If managed identity fails (permissions not assigned), the cell falls back to device code login.

In [ ]:
import subprocess, json as _j

# Check if already logged in
r = subprocess.run(['az', 'account', 'show'], capture_output=True, text=True)
if r.returncode == 0:
    acct = _j.loads(r.stdout)
    print(f'✅ Already logged in: {acct.get("user", {}).get("name", "unknown")}')
    print(f'   Subscription: {acct.get("name", "unknown")}')
else:
    # Try managed identity first (instant on Azure ML compute)
    r2 = subprocess.run(['az', 'login', '--identity'], capture_output=True, text=True)
    if r2.returncode == 0:
        acct = _j.loads(r2.stdout)[0]
        print(f'✅ Logged in via managed identity')
        print(f'   Subscription: {acct.get("name", "unknown")}')
    else:
        print('Managed identity not available. Using device code login...')
        !az login --use-device-code

## Step 2: Create Azure Function App

In [ ]:
!az functionapp create \
    --name {FUNC_APP_NAME} \
    --resource-group {RESOURCE_GROUP} \
    --storage-account {STORAGE_NAME} \
    --consumption-plan-location {LOCATION} \
    --runtime python \
    --runtime-version 3.11 \
    --functions-version 4 \
    --os-type Linux \
    --output table

print('\n✅ Function App created.')

**Expected output:** Function app details with `Succeeded` status.

## Step 3: Configure App Settings

Store connection details as app settings (not in code).

In [ ]:
!az functionapp config appsettings set \
    --name {FUNC_APP_NAME} \
    --resource-group {RESOURCE_GROUP} \
    --settings \
        AZURE_OPENAI_ENDPOINT="{OPENAI_ENDPOINT}" \
        AZURE_OPENAI_KEY="{OPENAI_KEY}" \
        AZURE_SEARCH_ENDPOINT="{SEARCH_ENDPOINT}" \
        AZURE_SEARCH_KEY="{SEARCH_KEY}" \
        AZURE_SEARCH_INDEX="banking-policies" \
    --output table

print('\n✅ App settings configured.')

## Step 4: Create Function Code

The function code implements the RAG pipeline as an HTTP-triggered Azure Function.

In [ ]:
func_dir = os.path.expanduser('~/banking-rag-func/rag_chat')
os.makedirs(func_dir, exist_ok=True)

# requirements.txt
with open(os.path.expanduser('~/banking-rag-func/requirements.txt'), 'w') as f:
    f.write('azure-functions\nopenai>=1.0.0\nazure-search-documents>=11.4.0\n')

# host.json
with open(os.path.expanduser('~/banking-rag-func/host.json'), 'w') as f:
    json.dump({'version':'2.0','extensionBundle':{'id':'Microsoft.Azure.Functions.ExtensionBundle','version':'[4.*, 5.0.0)'}}, f, indent=2)

# function.json
with open(os.path.join(func_dir, 'function.json'), 'w') as f:
    json.dump({'scriptFile':'__init__.py','bindings':[
        {'authLevel':'function','type':'httpTrigger','direction':'in','name':'req','methods':['post'],'route':'chat'},
        {'type':'http','direction':'out','name':'$return'}]}, f, indent=2)

# __init__.py
func_code = '''import azure.functions as func
import json, os
from openai import AzureOpenAI
from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizedQuery
from azure.core.credentials import AzureKeyCredential

openai_client = AzureOpenAI(azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_KEY"], api_version="2024-06-01")
search_client = SearchClient(endpoint=os.environ["AZURE_SEARCH_ENDPOINT"],
    index_name=os.environ.get("AZURE_SEARCH_INDEX", "banking-policies"),
    credential=AzureKeyCredential(os.environ["AZURE_SEARCH_KEY"]))

SYSTEM_PROMPT = """You are a SecureBank banking policy assistant.
Answer ONLY from context. Cite sources as [Source: filename]. Be concise."""

def main(req: func.HttpRequest) -> func.HttpResponse:
    try:
        body = req.get_json()
        question = body.get("question", "")
        if not question:
            return func.HttpResponse(json.dumps({"error":"Missing question"}), status_code=400, mimetype="application/json")
        vector = openai_client.embeddings.create(model="text-embedding-3-small", input=question).data[0].embedding
        results = list(search_client.search(search_text=question,
            vector_queries=[VectorizedQuery(vector=vector, k_nearest_neighbors=3, fields="content_vector")],
            select=["title","content","category","source_file"], top=3))
        context = "\\n\\n".join([f"[{d['source_file']}] {d['content']}" for d in results])
        messages = [{"role":"system","content":SYSTEM_PROMPT},
            {"role":"user","content":f"Context:\\n{context}\\n\\nQuestion: {question}"}]
        resp = openai_client.chat.completions.create(model="gpt-4o", messages=messages, max_tokens=150, temperature=0.1)
        return func.HttpResponse(json.dumps({"answer":resp.choices[0].message.content,
            "sources":[{"title":d["title"],"file":d["source_file"]} for d in results],
            "tokens":resp.usage.total_tokens}), mimetype="application/json")
    except Exception as e:
        return func.HttpResponse(json.dumps({"error":str(e)}), status_code=500, mimetype="application/json")
'''
with open(os.path.join(func_dir, '__init__.py'), 'w') as f:
    f.write(func_code)

print('✅ Function code created:')
print('   ~/banking-rag-func/requirements.txt')
print('   ~/banking-rag-func/host.json')
print('   ~/banking-rag-func/rag_chat/function.json')
print('   ~/banking-rag-func/rag_chat/__init__.py')

## Step 5: Deploy Function

Deploy the function to Azure. If `func` CLI is not available, the code shows manual deployment steps.

In [ ]:
import shutil

func_base = os.path.expanduser('~/banking-rag-func')
print('🚀 Deploying function to Azure...')
print()

# Try func CLI first, fall back to zip deploy
result = subprocess.run(['which', 'func'], capture_output=True)
if result.returncode == 0:
    !cd ~/banking-rag-func && func azure functionapp publish {FUNC_APP_NAME} --python
else:
    # Zip deploy
    shutil.make_archive('/tmp/banking-rag-func', 'zip', func_base)
    !az functionapp deployment source config-zip \
        --name {FUNC_APP_NAME} \
        --resource-group {RESOURCE_GROUP} \
        --src /tmp/banking-rag-func.zip \
        --output table

FUNC_URL = f'https://{FUNC_APP_NAME}.azurewebsites.net/api/chat'
print(f'\n📋 Endpoint: {FUNC_URL}')

## Step 6: Test the Endpoint

In [ ]:
# Get function key
result = subprocess.run(['az','functionapp','keys','list','--name',FUNC_APP_NAME,
    '--resource-group',RESOURCE_GROUP,'--query','functionKeys.default','-o','tsv'],
    capture_output=True, text=True)
FUNC_KEY = result.stdout.strip()

# Test 1: Simple question
print('--- Test 1: Simple question ---')
test_url = f'{FUNC_URL}?code={FUNC_KEY}'
data = json.dumps({'question':'What is the wire transfer limit for personal accounts?'}).encode()
try:
    req = urllib.request.Request(test_url, data=data, headers={'Content-Type':'application/json'})
    with urllib.request.urlopen(req, context=ctx, timeout=30) as resp:
        result = json.loads(resp.read())
    print(json.dumps(result, indent=2))
except Exception as e:
    print(f'  Function may still be warming up. Try again in 30s. Error: {e}')

# Test 2: Multi-turn
print('\n--- Test 2: Multi-turn conversation ---')
data = json.dumps({'question':'What about business accounts?',
    'chat_history':[{'user':'What is the wire transfer limit?','assistant':'$50,000 domestic for personal.'}]}).encode()
try:
    req = urllib.request.Request(test_url, data=data, headers={'Content-Type':'application/json'})
    with urllib.request.urlopen(req, context=ctx, timeout=30) as resp:
        result = json.loads(resp.read())
    print(json.dumps(result, indent=2))
except Exception as e:
    print(f'  Error: {e}')

## Step 7: Save Endpoint Configuration

In [ ]:
config_path = os.path.expanduser('~/lab-config.env')
with open(config_path, 'a') as f:
    f.write(f'\nexport FUNC_APP_NAME="{FUNC_APP_NAME}"\n')
    f.write(f'export FUNC_URL="{FUNC_URL}"\n')
    f.write(f'export FUNC_KEY="{FUNC_KEY}"\n')
print('✅ Endpoint config saved to ~/lab-config.env')

## ✅ Lab 7 Complete

**What you accomplished:**
- Created Azure Function with RAG pipeline
- Configured secure app settings
- Deployed serverless REST API endpoint
- Tested with single and multi-turn queries

**Next:** Open `promptflow_lab8_monitoring.ipynb`


> 🧹 **Cleanup:** Run the cleanup cell in **Lab 9** when all labs are complete to delete all resources.
---

© 2026 Great Learning. All rights reserved.